# M5: M2 표현 + M4 손실 결합 (Dunnhumby)

고정된 M2 CLV 표현과 M4 CLV 조건부 hard-negative 손실을 하나의 학습 루프에서 결합합니다. 개발용 seed 42 한 번만 실행하며 final test와 holdout은 만들지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = 'b69304c4478b8ab785464703b950e8ea370880ef'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_embedding_hard_negative import (
    configure_m5_run,
    preflight_summary,
    run_m5_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1(K=5), M2, M4, M5, M5 CLV 순열')
display(result_df)
print('2) 대조군별 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) M2×M4 상호작용')
display(pd.DataFrame(result_df.attrs['interaction']))
print('4) 사전 판정')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))